In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from song_cleaning.key_mode_geometry import build_geometry_training_frame, decode_xyz_to_key_mode
from song_cleaning.key_mode_metrics import compute_key_mode_metrics

DATA_PATH = ROOT / "liveness_final.parquet"
FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]
MODEL_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 600,
    "max_depth": 6,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}


In [5]:
columns_to_load = sorted(set(FEATURE_POOL + ["key", "mode", "track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
float64_columns = df.select_dtypes(include=["float64"]).columns
df[float64_columns] = df[float64_columns].astype("float32")
df[["key", "mode"]].isna().sum()


key     446161
mode         0
dtype: int64

In [6]:
X_observed, xyz_targets = build_geometry_training_frame(df, FEATURE_POOL)
y_key = df.loc[df["key"].notna() & df["mode"].notna(), "key"].astype("int32")
y_mode = df.loc[df["key"].notna() & df["mode"].notna(), "mode"].astype("int32")

X_train, X_test, y_x_train, y_x_test, y_y_train, y_y_test, y_z_train, y_z_test, y_key_train, y_key_test, y_mode_train, y_mode_test = train_test_split(
    X_observed,
    xyz_targets["x"],
    xyz_targets["y"],
    xyz_targets["z"],
    y_key,
    y_mode,
    test_size=0.2,
    random_state=42,
)

len(X_train), len(X_test)


(2342990, 585748)

In [7]:
x_model = XGBRegressor(**MODEL_PARAMS)
y_model = XGBRegressor(**MODEL_PARAMS)
z_model = XGBRegressor(**MODEL_PARAMS)

x_model.fit(X_train, y_x_train)
y_model.fit(X_train, y_y_train)
z_model.fit(X_train, y_z_train)

predicted_x = x_model.predict(X_test).astype(np.float32)
predicted_y = y_model.predict(X_test).astype(np.float32)
predicted_z = z_model.predict(X_test).astype(np.float32)

decoded = decode_xyz_to_key_mode(predicted_x, predicted_y, predicted_z)
metrics = compute_key_mode_metrics(
    true_keys=y_key_test,
    true_modes=y_mode_test,
    predicted_keys=decoded["predicted_key"],
    predicted_modes=decoded["predicted_mode"],
)
metrics


{'key_accuracy': 0.08967678933602846,
 'mode_accuracy': 0.6521848986253475,
 'joint_accuracy': 0.053140599711821467,
 'mean_key_distance': 2.835803109869773,
 'median_key_distance': 3.0,
 'within_1_semitone': 0.27495783169554144,
 'within_2_semitones': 0.4553767149012886}

In [8]:
missing_df = df[df["key"].isna() | df["mode"].isna()].copy()
X_missing = missing_df[FEATURE_POOL]

missing_x = x_model.predict(X_missing).astype(np.float32)
missing_y = y_model.predict(X_missing).astype(np.float32)
missing_z = z_model.predict(X_missing).astype(np.float32)

decoded_missing = decode_xyz_to_key_mode(missing_x, missing_y, missing_z)
imputed_rows = missing_df[["track_id", "artist_name", "track_name"]].copy()
imputed_rows["key"] = decoded_missing["predicted_key"].to_numpy()
imputed_rows["mode"] = decoded_missing["predicted_mode"].to_numpy()
imputed_rows.head()


,track_id,artist_name,track_name,key,mode
3,NaN,yungmanny,waldo,11,0
9,NaN,creature feature,bound and gagged,10,1
20,NaN,three 6 mafia,south memphis bitch,11,1
23,NaN,roxy music,for your pleasure,10,1
28,NaN,stick figure,angels among us,11,0
